In [ ]:
import os, json, re, time
from collections import Counter
import requests

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4o-mini") 


def extract_fields_openrouter(text_en: str, api_key: str) -> dict:
    sys = (
        "Extract structured clinical fields from the user's English query text. "
        "Return ONLY valid JSON with keys: age (int or null), sex (\"male\"/\"female\"/\"other\" or null), "
        "chief_concern (string or null), history (string or null), diagnosis_guess (string or null). "
        "chief_concern should describe the current issue/symptoms; history is past dx/PMH/meds/allergies/preceding illness, etc. "
        "diagnosis_guess is only if the user explicitly suggests/was told a specific diagnosis in the text."
    )
    user = f"TEXT:\n{text_en}\n\nReturn JSON only."

    r = requests.post(
        OPENROUTER_URL,
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
            "HTTP-Referer": os.getenv("OPENROUTER_HTTP_REFERER", "http://localhost"),
            "X-Title": os.getenv("OPENROUTER_APP_TITLE", "jsonl-variant-gen"),
        },
        json={
            "model": MODEL,
            "messages": [{"role": "system", "content": sys}, {"role": "user", "content": user}],
            "temperature": 0,
        },
        timeout=60,
    )
    r.raise_for_status()
    content = r.json()["choices"][0]["message"]["content"].strip()
    content = re.sub(r"^```(?:json)?\s*|\s*```$", "", content).strip()
    return json.loads(content)



_STOP = {
    "a", "an", "the", "and", "or", "to", "of", "in", "on", "for", "with", "without",
    "seems", "likely", "possible", "typical", "consistent", "diagnosis"
}

def _normalize_dx(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    s = s.strip(" .,:;!?\"'`()[]{}")
    return s

def _clean_candidate(s: str) -> str:
    s = _normalize_dx(s)
    if not s:
        return ""
    # remove trailing explanatory clause often added in responses
    s = re.split(r"\b(seems|likely|possible|consistent|and|but)\b", s, maxsplit=1, flags=re.I)[0].strip()
    s = _normalize_dx(s)

    # avoid super-long sentences; keep "diagnosis-like" short phrases
    if len(s) > 80:
        s = s[:80].rsplit(" ", 1)[0].strip()

    # remove leading stopwords
    toks = s.split()
    while toks and toks[0].lower() in _STOP:
        toks = toks[1:]
    s = " ".join(toks).strip()
    return s

def infer_dx_guess_from_obj(obj: dict):
    """
    Pick the most frequent short diagnosis-like phrase from any responses.
    Prefers English, but will fall back to zh/es if that's all you have.
    """
    responses = obj.get("responses") or []
    cands = []

    for r in responses:
        for k in ("content_en", "content_zh", "content_es"):
            txt = (r.get(k) or "").strip()
            if not txt:
                continue

            # If it's like "Typical Psoriasis" or "Psoriasis seems ...", try to extract a front diagnosis phrase
            # Heuristic: use first clause up to punctuation.
            front = re.split(r"[.\n;:，。]", txt, maxsplit=1)[0]
            front = _clean_candidate(front)

            # Extra heuristic: if it starts with "typical X" -> keep X
            m = re.match(r"(?i)typical\s+(.+)$", front)
            if m:
                front = _clean_candidate(m.group(1))

            if front:
                cands.append(front)

    if not cands:
        return None

    # Vote by normalized lowercase, but keep a representative original casing for output
    norm = [c.lower() for c in cands]
    top_norm, _ = Counter(norm).most_common(1)[0]
    # choose the shortest representative among those (often the clean dx label)
    reps = [c for c in cands if c.lower() == top_norm]
    reps.sort(key=lambda x: (len(x), x))
    return reps[0] if reps else None

def infer_dx_openrouter_from_responses(obj: dict, api_key: str):
    """
    Use OpenRouter to infer a best-guess diagnosis label from any responses in the input.
    Returns a short string like "Psoriasis" or None.
    """
    responses = obj.get("responses") or []
    if not responses:
        return None

    # Build a compact prompt of the responses (en/zh/es). Keep it short to reduce cost.
    lines = []
    for i, r in enumerate(responses, 1):
        en = (r.get("content_en") or "").strip()
        parts = []
        if en: parts.append(f"en: {en}")
        if parts:
            lines.append(f"{i}. " + " | ".join(parts))

    if not lines:
        return None

    sys = (
        "You are given multiple clinician responses about a case. "
        "Infer the most likely diagnosis label from these responses. "
        "Return ONLY valid JSON: {\"dx_guess\": string or null}. "
        "If multiple appear, choose the most common / consensus. "
        "Keep dx_guess short (<= 6 words), no punctuation, no explanation."
    )
    user = "RESPONSES:\n" + "\n".join(lines) + "\n\nReturn JSON only."

    r = requests.post(
        OPENROUTER_URL,
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
            "HTTP-Referer": os.getenv("OPENROUTER_HTTP_REFERER", "http://localhost"),
            "X-Title": os.getenv("OPENROUTER_APP_TITLE", "jsonl-variant-gen"),
        },
        json={
            "model": MODEL,
            "messages": [{"role": "system", "content": sys}, {"role": "user", "content": user}],
            "temperature": 0,
        },
        timeout=60,
    )
    r.raise_for_status()
    content = r.json()["choices"][0]["message"]["content"].strip()
    content = re.sub(r"^```(?:json)?\s*|\s*```$", "", content).strip()

    try:
        dx = json.loads(content).get("dx_guess")
    except Exception:
        return None

    dx = _normalize_dx(dx) if isinstance(dx, str) else None
    if dx:
        dx = _clean_candidate(dx)  
    return dx or None


# --- variants -------------------------------------------------------------------

def build_variants(base_q: str, age: int, sex: str, concern: str, history: str, dx_guess):
    demo = f"I am a {age} year old {sex}."
    concern = (concern or "").rstrip().rstrip(".") + "."
    history = (history or "").rstrip().rstrip(".") + "."

    v1 = base_q
    v2 = f"{demo} {base_q}"
    v3 = f"{concern} {demo} {base_q}"
    v4 = f"{history} {concern} {demo} {base_q}"
    
    if dx_guess:
        dx_guess = dx_guess.strip()
        v5 = f"{history} {concern} {demo} {base_q} I think it is {dx_guess}."
    else:
        # still generate v5 even if we couldn't infer
        v5 = f"{history} {concern} {demo} {base_q}"

    return [v1, v2, v3, v4, v5]


def main(in_jsonl: str):
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise SystemExit("Missing OPENROUTER_API_KEY env var.")

    outs = [open(f"{in_jsonl}.v{i}.jsonl", "a", encoding="utf-8") for i in range(1, 6)]
    kept = 0
    base_q = "What is this skin condition?"

    with open(in_jsonl, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            print(f"Processing line {line_no}...", end="\r")
            if line_no:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except Exception:
                    continue

                text_en = obj.get("query_content_en") or ""
                if not text_en.strip():
                    continue

                try:
                    fields = extract_fields_openrouter(text_en, api_key)
                except Exception:
                    continue

                age = fields.get("age")
                sex = fields.get("sex")
                concern = fields.get("chief_concern")
                history = fields.get("history")

                # NEW: dx comes from extractor if present, else infer from responses
                dx_guess = fields.get("diagnosis_guess")
                dx_guess = _normalize_dx(dx_guess) if dx_guess else None
                if not dx_guess:
                    try:
                        dx_guess = infer_dx_openrouter_from_responses(obj, api_key)
                    except Exception:
                        continue

                # eligibility: no longer require dx_guess
                if not (isinstance(age, int) and sex in ("male", "female", "other") and concern and history):
                    continue

                variants = build_variants(base_q, age, sex, concern, history, dx_guess)

                print(
                    f"line {line_no}: age={age}, sex={sex}, "
                    f"concern={concern!r}, history={history!r}, dx_guess={dx_guess!r}"
                )

                for i, out in enumerate(outs):
                    o = dict(obj)
                    o["query_content_en"] = variants[i]
                    out.write(json.dumps(o, ensure_ascii=False) + "\n")

                kept += 1
                time.sleep(float(os.getenv("OPENROUTER_SLEEP_SEC", "0")))

    for out in outs:
        out.close()
        

    print(f"done. kept={kept}")
    print("wrote:", ", ".join([f"{in_jsonl}.v{i}.jsonl" for i in range(1, 6)]))


In [2]:
main("/Users/USER/Documents/DermaEval_Thesis/train.jsonl")

line 8: extracted fields={'age': 6, 'sex': 'female', 'chief_concern': 'skin lesions behind right eyebrow and right ear with slight pain', 'history': 'treated with Jumbizhi and Huanglian; no fever or lesions on other parts of the body', 'diagnosis_guess': 'vesicular dermatitis'} -> variants=['What is this skin condition?', 'I am a 6 year old female. What is this skin condition?', 'skin lesions behind right eyebrow and right ear with slight pain. I am a 6 year old female. What is this skin condition?', 'treated with Jumbizhi and Huanglian; no fever or lesions on other parts of the body. skin lesions behind right eyebrow and right ear with slight pain. I am a 6 year old female. What is this skin condition?', 'treated with Jumbizhi and Huanglian; no fever or lesions on other parts of the body. skin lesions behind right eyebrow and right ear with slight pain. I am a 6 year old female. What is this skin condition? I think it is vesicular dermatitis.']
line 13: extracted fields={'age': 60, 's

In [13]:
main("/Users/USER/Documents/DermaEval_Thesis/train_2.jsonl")

line 439: age=28, sex=female, concern='bumps at the junction of the inner and outer part of the right foot that reappear every April', history='has been developing these bumps for the past three years; many medications have been used for treatment', dx_guess='athletes foot'
line 444: age=50, sex=male, concern='sweat stains and Malassezia folliculitis', history='white spots on the skin at the sternum', dx_guess='Pityriasis Versicolor'
line 446: age=56, sex=female, concern='finger chapping and significant pain', history='suffering from finger chapping for 2 years, normal diet, no bad habits, washes hands daily with a decoction of mugwort', dx_guess='eczema'
line 448: age=18, sex=male, concern='Periorbital rash for three years, unilateral', history='Oral antihistamines or topical steroid ointments are effective, but the rash often recurs after stopping the medication.', dx_guess='neurodermatitis'
line 451: age=38, sex=male, concern='blisters on the toes of both feet, thick skin, slight it

In [ ]:
# generate VERSION 6
import json
from pathlib import Path

INPUT_JSONL = "/Users/USER/Documents/DermaEval_Thesis/train_2.jsonl.v1.jsonl"
OUTPUT_JSONL = "/Users/USER/Documents/DermaEval_Thesis/train_2.jsonl.v8.jsonl"

FIXED_QUERY_CONTENT_EN = (
    """
    Please review the attached medical images and describe what condition you believe is most likely present. 
    Explain your reasoning in detail, mention possible alternative diagnoses, and note any additional information that would help 
    confirm your assessment."
    """
)

def transform_file(in_path: str, out_path: str):
    in_path = Path(in_path)
    out_path = Path(out_path)

    n_in = 0
    n_out = 0
    n_bad = 0

    with in_path.open("r", encoding="utf-8") as fin, out_path.open("w", encoding="utf-8") as fout:
        for line in fin:
            line = line.strip()
            if not line:
                continue
            n_in += 1

            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                n_bad += 1
                continue

            # Replace / set the field
            obj["query_content_en"] = FIXED_QUERY_CONTENT_EN

            # Write as JSONL (one object per line)
            fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
            n_out += 1

    print(f"Read: {n_in}, Written: {n_out}, Bad lines skipped: {n_bad}")
    print(f"Saved → {out_path}")

if __name__ == "__main__":
    transform_file(INPUT_JSONL, OUTPUT_JSONL)

Read: 228, Written: 228, Bad lines skipped: 0
Saved → /Users/USER/Documents/DermaEval_Thesis/train_2.jsonl.v6.jsonl


## Generate versions 7 (close wrong diagnosis) and version 8 (far wrong diagnosis )

In [ ]:
import os, json, re, time
import requests

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4o-mini")

# ---------------- OpenRouter helper ----------------

def _post_openrouter(messages, api_key: str, temperature: float = 0.0, timeout: int = 60) -> str:
    r = requests.post(
        OPENROUTER_URL,
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
            "HTTP-Referer": os.getenv("OPENROUTER_HTTP_REFERER", "http://localhost"),
            "X-Title": os.getenv("OPENROUTER_APP_TITLE", "jsonl-v7v8-gen"),
        },
        json={
            "model": MODEL,
            "messages": messages,
            "temperature": temperature,
        },
        timeout=timeout,
    )
    r.raise_for_status()
    content = r.json()["choices"][0]["message"]["content"].strip()
    content = re.sub(r"^```(?:json)?\s*|\s*```$", "", content).strip()
    return content

# ---------------- diagnosis string utils ----------------

_STOP = {
    "a", "an", "the", "and", "or", "to", "of", "in", "on", "for", "with", "without",
    "seems", "likely", "possible", "typical", "consistent", "diagnosis"
}

def _normalize_dx(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    s = s.strip(" .,:;!?\"'`()[]{}")
    return s

def _clean_candidate(s: str) -> str:
    s = _normalize_dx(s)
    if not s:
        return ""
    s = re.split(r"\b(seems|likely|possible|consistent|and|but)\b", s, maxsplit=1, flags=re.I)[0].strip()
    s = _normalize_dx(s)
    if len(s) > 80:
        s = s[:80].rsplit(" ", 1)[0].strip()
    toks = s.split()
    while toks and toks[0].lower() in _STOP:
        toks = toks[1:]
    return " ".join(toks).strip()

# ---------------- parse/replace "I think it is ..." ----------------

# Captures dx in: "I think it is Beriberi." / "I think it is Beriberi"
DX_PAT = re.compile(r"\bI\s+think\s+it\s+is\s+(?P<dx>.+?)(?:\.\s*|$)", re.I)

def extract_dx_from_query(query_en: str):
    if not query_en:
        return None
    m = DX_PAT.search(query_en)
    if not m:
        return None
    dx = _clean_candidate(m.group("dx"))
    return dx or None

def replace_dx_in_query(query_en: str, new_dx: str) -> str:
    """
    Replace the dx inside the first occurrence of 'I think it is ...'
    Preserve everything else.
    """
    def _repl(match: re.Match) -> str:
        # Ensure we end with a period
        return f"I think it is {new_dx}."

    # Replace first occurrence only
    out, n = DX_PAT.subn(_repl, query_en, count=1)
    if n == 0:
        # If pattern not found, append at end
        out = (query_en.rstrip() + " " if query_en.strip() else "") + f"I think it is {new_dx}."
    return out

# ---------------- OpenRouter: close + far wrong generation ----------------

def dx_close_and_far_wrong(dx_correct: str, api_key: str) -> dict:
    """
    From the *current* dx label found in v5, generate:
      - dx_close: close differential / same family
      - dx_far: very far / bad guess, different family
    """
    dx_correct = _clean_candidate(dx_correct)
    if not dx_correct:
        return {"dx_close": None, "dx_far": None}

    sys = (
        "You will be given a diagnosis label. Create two WRONG alternatives:\n"
        "1) dx_close: plausible wrong differential in the same family/category.\n"
        "2) dx_far: a very bad guess from a completely different family/category.\n\n"
        "Return ONLY valid JSON: {\"dx_close\": string or null, \"dx_far\": string or null}\n"
        "Constraints:\n"
        "- Each <= 6 words\n"
        "- No punctuation, no explanation\n"
        "- Must not equal the original\n"
        "- dx_far must be clearly unrelated"
    )
    user = f"DX: {dx_correct}\nReturn JSON only."

    content = _post_openrouter(
        [{"role": "system", "content": sys}, {"role": "user", "content": user}],
        api_key=api_key,
        temperature=0.0,
    )

    try:
        out = json.loads(content)
    except Exception:
        return {"dx_close": None, "dx_far": None}

    dx_close = out.get("dx_close")
    dx_far = out.get("dx_far")

    dx_close = _clean_candidate(dx_close) if isinstance(dx_close, str) else None
    dx_far = _clean_candidate(dx_far) if isinstance(dx_far, str) else None

    if dx_close and dx_close.lower() == dx_correct.lower():
        dx_close = None
    if dx_far and dx_far.lower() == dx_correct.lower():
        dx_far = None

    return {"dx_close": dx_close or None, "dx_far": dx_far or None}

# ---------------- main ----------------

def main(v5_jsonl: str):
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise SystemExit("Missing OPENROUTER_API_KEY env var (do not hardcode secrets).")

    out_v7 = open(f"{v5_jsonl}_close_v7.jsonl", "w", encoding="utf-8")
    out_v8 = open(f"{v5_jsonl}_far_v8.jsonl", "w", encoding="utf-8")

    # cache per dx label so repeated dx doesn’t cost extra
    cache: dict[str, tuple[str | None, str | None]] = {}

    kept = 0
    skipped_no_dx = 0

    with open(v5_jsonl, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)
            except Exception:
                continue

            q = obj.get("query_content_en") or ""
            dx = extract_dx_from_query(q)
            if not dx:
                skipped_no_dx += 1
                continue

            key = dx.lower()
            if key in cache:
                dx_close, dx_far = cache[key]
            else:
                d = dx_close_and_far_wrong(dx, api_key)
                dx_close, dx_far = d.get("dx_close"), d.get("dx_far")
                cache[key] = (dx_close, dx_far)

            # If model fails, fall back to original dx (still write outputs)
            dx_close_final = dx_close or dx
            dx_far_final = dx_far or dx

            obj6 = dict(obj)
            obj7 = dict(obj)

            obj6["query_content_en"] = replace_dx_in_query(q, dx_close_final)
            obj7["query_content_en"] = replace_dx_in_query(q, dx_far_final)

            out_v7.write(json.dumps(obj6, ensure_ascii=False) + "\n")
            out_v8.write(json.dumps(obj7, ensure_ascii=False) + "\n")

            kept += 1

            # optional throttle
            time.sleep(float(os.getenv("OPENROUTER_SLEEP_SEC", "0")))

            if line_no % 50 == 0:
                print(f"Processed {line_no} lines... kept={kept}, skipped_no_dx={skipped_no_dx}", end="\r")

    out_v7.close()
    out_v8.close()

    print()
    print(f"done. kept={kept}, skipped_no_dx={skipped_no_dx}")
    print(f"wrote: {v5_jsonl}_close_v7.jsonl")
    print(f"wrote: {v5_jsonl}_far_v8.jsonl")


In [8]:
main("/Users/USER/Documents/Dermatology-Evaluation/DermDetail/Dataset/train_2.jsonl.v5.jsonl")

Processed 200 lines... kept=197, skipped_no_dx=3
done. kept=226, skipped_no_dx=3
wrote: /Users/USER/Documents/Dermatology-Evaluation/DermDetail/Dataset/train_2.jsonl.v5.jsonl_close_v7.jsonl
wrote: /Users/USER/Documents/Dermatology-Evaluation/DermDetail/Dataset/train_2.jsonl.v5.jsonl_far_v8.jsonl
